# RAG: Retrieval-Augmented Generation

- **R(Retrieval, 검색)**: 외부에 있는 컨텐츠를 추출하는 단계
- **A(Augmentation, 증강/확장)**: Retrieval 단계에서 추출한 컨텐츠를 전달할 수 있도록 수정하는 단계
- **G(Generation, 생성)**: LLM을 사용해서 최종 결과를 생성하는 단계

```mermaid
flowchart TD
    subgraph "1. Document Processing"
        A[Documents] --> B[Split Text into Chunks]
        B --> C1[Chunk-1]
        B --> C2[Chunk-2]
        B --> C3[Chunk-n]
    end
    
    subgraph "2. Document Embedding"
        EM1{{Embedding Model}}
        C1 & C2 & C3 --> EM1
        EM1 --> D1[Embedding-1] & D2[Embedding-2] & D3[Embedding-3]
    end
    
    subgraph "3. Indexing"
        D1 & D2 & D3 --> E[(VectorDB)]
    end
    
    subgraph "4. Query Processing"
        F[Query] --> EM2{{Embedding Model}}
        EM2 --> G[Query Embedding]
    end
    
    subgraph "5. Retrieval"
        G -->|Similarity Search| E
        E -->|Top-K Retrieval| H[Relevant Chunks]
    end
    
    subgraph "6. Context Formation"
        H --> I[Query + Relevant Chunks]
    end
    
    subgraph "7. Generation"
        I --> J[LLM]
        J --> K[Response]
    end
    
    F --> I
```

## Programming stage 관점에서 살펴보면...

### 1. Data Ingestion (데이터 수집)
- PDF를 parsing 한 후에 text를 추출
- text chunking 수행
  - parsing된 문서를 LLM이 처리 가능한 단위로 나누는 과정
- DB 준비
  - 여기서 자주 사용되는게 흔히들 들었던 Vector DB
  - Why? query와의 유사한 chunk를 찾기 위해서. 
    - 즉, input과 chunk 간의 similairty 계산을 하기 위함.
- parsing된 데이터(chunking 된 data)로 database를 채우기


### 2. Retrieval (검색)
- user query를 입력으로 GET
- 저장된 data들 전체에 걸쳐서 similarity search 수행
- 가장 관련성이 높은 (유사도가 높게 나온) 정보들을 추출


### 3. Augmentation (증강/확장)
- 추출된 data에서 관련된 부분들을 통합해 prompt를 보강(augment)
- prompt engineering을 통해 prompt를 조정해서 clarity(명확성)과 context(문맥)을 최적화



### 4. Generation (생성)
- 증강된(향상된, 개선된, 확장된) prompt를 사용, LLM을 활용해서 최종 response를 생성

In [1]:
from importlib.metadata import version

packages = ["numpy",
            "scipy",
            "torch",
            "sentence-transformers",
            "wikipedia-api",
            "openai",
            "rich",
            "pypdf2",
            "gradio"
            ]

for package in packages:
    try:
        pkg_version = version(package)
    except Exception as e:
        pkg_version = f"Error retrieving version: {e}"
    print(f"{package}: {pkg_version}")

numpy: 2.1.3
scipy: 1.14.1
torch: 2.4.1
sentence-transformers: 5.3.0
wikipedia-api: 0.14.0
openai: 2.30.0
rich: 14.3.1
pypdf2: 3.0.1
gradio: 6.11.0


In [2]:
import re, os, openai

from rich import print # 이쁜 output
from sentence_transformers import SentenceTransformer
from wikipediaapi import Wikipedia

import numpy as np
import textwrap
import PyPDF2

c:\Users\skdbs\.conda\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data 준비

- tutorial에서 제공하는 preprocessed data (.md)
- [attention_is_all_you_need.md](https://github.com/adithya-s-k/AI-Engineering.academy/blob/main/archives/data/md/attention_is_all_you_need.md) 필요.

In [3]:
def load_document(file_path):
    """
    주어진 file path에서 문서 load (PDF, text)
    """

    _, file_extension = os.path.splitext(file_path)

    if file_extension.lower() == ".pdf":
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""

            # pdf에서 text 추출
            for page in pdf_reader.pages:
                text += page.extract_text()

    elif file_extension.lower() == ".txt":
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()

    elif file_extension.lower() == ".md":
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
    
    else:
        raise ValueError("Unsupported file format. Provide a PDF or text file.")
    
    return text

In [4]:
data = load_document(os.getcwd() + '/rag_dataset/attention_is_all_you_need.md')
print(data[:500])

# Attention Is All You Need

# Authors

Ashish Vaswani*

Noam Shazeer*

Niki Parmar*

Jakob Uszkoreit*

Google Brain

avaswani@google.com

noam@google.com

nikip@google.com

usz@google.com

Llion Jones*

Aidan N. Gomez* †

Łukasz Kaiser*

Google Research

University of Toronto

llion@google.com

aidan@cs.toronto.edu

lukaszkaiser@google.com

Illia Polosukhin* ‡

illia.polosukhin@gmail.com

# Abstract

The dominant sequence transduction models are based on complex recurrent or convolutional neura

### Chunking

- load 한 문서(pdf, md, txt)를 LLM이 처리 가능한 단위로 나누는 작업

In [5]:
def chunk_text(text, chunk_size=100, overlap=20):
    """
    단어 수와 중복되는 단어를 기준으로 text를 chunk로 분할
    """

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

In [6]:
chunked_data = chunk_text(data)

# chunk가 어떻게 생성되었는지 확인
for i, chunk in enumerate(chunked_data):
    print(f"Chunk {i + 1}:")
    print(chunk)
    print("=" * 50)

    if i == 4:
        break

Chunk 1:

# Attention Is All You Need # Authors Ashish Vaswani* Noam Shazeer* Niki Parmar* Jakob Uszkoreit* Google Brain 
avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones* Aidan N. Gomez* † Łukasz Kaiser* 
Google Research University of Toronto llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia 
Polosukhin* ‡ illia.polosukhin@gmail.com # Abstract The dominant sequence transduction models are based on complex 
recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also 
connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the 
Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. 
Experiments

==================================================

Chunk 2:

propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with 
recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior
in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 
BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including 
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new 
single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the
training

...

and implemented the first Transformer models and has been crucially involved in every aspect of this work. Noam 
proposed scaled dot-product attention, multi-head attention and the parameter-free position representation and 
became the other person involved in nearly every detail. Niki designed, implemented, tuned and evaluated countless 
model variants in our original codebase and tensor2tensor. Llion also experimented with novel model variants, was 
responsible for our initial codebase, and efficient inference and visualizations. Lukasz and Aidan spent countless 
long days designing various parts of and implementing tensor2tensor, replacing our earlier codebase, greatly 
improving results and massively accelerating our research. † Work

==================================================

Chunk 5:

various parts of and implementing tensor2tensor, replacing our earlier codebase, greatly improving results and 
massively accelerating our research. † Work performed while at Google Brain. ‡ Work performed while at Google 
Research. # Conference Information 31st Conference on Neural Information Processing Systems (NIPS 2017), Long 
Beach, CA, USA. --- # 1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7]
neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since 
continued to push

==================================================

~~위 부분은 example에서 안쓰는 듯;; 그냥 .md 따라가기로 함.~~

### Embedding model 준비

- chunk로 나눈 text들을 vector로 표현해야 LLM에 fed할 수 있음
- 따라서, well-made pretrained model을 사용함
  - NLP field에선 BERT같은 pretrained model을 이용해 text embedding을 생성하는 것이 일반적.

In [7]:
model = SentenceTransformer("Alibaba-NLP/gte-base-en-v1.5", trust_remote_code=True)

Loading weights: 100%|██████████| 135/135 [00:00<00:00, 592.10it/s, Materializing param=encoder.layer.11.mlp_ln.weight]            


In [8]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'NewModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

### Text content 준비
- Knowledge base를 준비하기 위해 wikipedia API를 활용

In [9]:
from wikipediaapi import Wikipedia

wiki = Wikipedia('RAGBot/0.0', 'en')
doc = wiki.page('Hayao_Miyazaki').text
paragraphs = doc.split('\n\n')          # chunking

- 문서를 chunking하는 방법은 많지만, 예상대로 동작하지 않는 경우가 많음.
- 따라서 여기선 knowledge base를 검토해서, paragraph 단위로 나눔.
- chunk가 어떻게 생성되었는지 한번 보면...

In [10]:
import textwrap

for i, p in enumerate(paragraphs):
    if i == 3:
        break
    wrapped_text = textwrap.fill(p, width=100)

    print("-----------------------------------------------------------------")
    print(wrapped_text)
    print("-----------------------------------------------------------------")

-----------------------------------------------------------------

Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; ; born January 5, 1941) is a Japanese
animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary
chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful
storyteller and creator of Japanese animated feature films, and is widely regarded as one of the
most accomplished filmmakers in the history of animation. Born in Tokyo City, Miyazaki expressed
interest in manga and animation from an early age. He joined Toei Animation in 1963, working as an
inbetween artist and key animator on films like Gulliver's Travels Beyond the Moon (1965), Puss in
Boots (1969), and Animal Treasure Island (1971), before moving to A-Pro in 1971, where he co-
directed Lupin the Third Part I (1971–1972) alongside Isao Takahata. After moving to Zuiyō Eizō
(later Nippon Animation) in 1973, Miyazaki worked as an animator on World Masterpiece Theater and
directed the television series Future Boy Conan (1978). He joined Tokyo Movie Shinsha in 1979 to
direct his first feature film The Castle of Cagliostro (1979) and the television series Sherlock
Hound (1984–1985). He wrote and illustrated the manga Nausicaä of the Valley of the Wind (1982–1994)
and directed the 1984 film adaptation produced by Topcraft. Miyazaki co-founded Studio Ghibli in
1985, writing and directing films such as Laputa: Castle in the Sky (1986), My Neighbor Totoro
(1988), Kiki's Delivery Service (1989), and Porco Rosso (1992), which were met with critical and
commercial success in Japan. Miyazaki's Princess Mononoke (1997) was the first animated film to win
the Japan Academy Film Prize for Picture of the Year and briefly became the highest-grossing film in
Japan; its Western distribution increased Ghibli's worldwide popularity and influence. Spirited Away
(2001) became Japan's highest-grossing film and won the Academy Award for Best Animated Feature; it
is frequently ranked among the greatest films of the 21st century. Miyazaki's later films—Howl's
Moving Castle (2004), Ponyo (2008), and The Wind Rises (2013)—also enjoyed critical and commercial
success. He retired from feature films in 2013 but later returned to make The Boy and the Heron
(2023), which won the Academy Award for Best Animated Feature. Miyazaki's works are frequently
subject to scholarly analysis and have been characterized by the recurrence of themes such as
humanity's relationship with nature and technology, the importance of art and craftsmanship, and the
difficulty of maintaining a pacifist ethic in a violent world. His protagonists are often strong
girls or young women, and several of his films present morally ambiguous antagonists with redeeming
qualities. Miyazaki's works have been highly praised and awarded; he was named a Person of Cultural
Merit for outstanding cultural contributions in 2012, received the Academy Honorary Award for his
impact on animation and cinema in 2014, and the Ramon Magsaysay Award in 2024. Miyazaki has
frequently been cited as an inspiration for numerous animators, directors, and writers.

-----------------------------------------------------------------

-----------------------------------------------------------------

Early life Hayao Miyazaki was born on January 5, 1941, in the town Akebono-cho in Hongō, Tokyo City,
Empire of Japan, the second of four sons. His father, Katsuji Miyazaki (born 1915), was the director
of Miyazaki Airplane, his brother's company, which manufactured rudders for fighter planes during
World War II. The business allowed his family to remain affluent during Miyazaki's early life.
Miyazaki's father enjoyed purchasing paintings and demonstrating them to guests, but otherwise had
little known artistic understanding. He was in the Imperial Japanese Army around 1940, discharged
and lectured about disloyalty after declaring to his commanding officer that he wished not to fight
because of his wife and young child. According to Miyazaki, his father often told him about his
exploits, claiming he continued to attend nightclubs after turning 70. Katsuji Miyazaki died on
March 18, 1993. After his death, Miyazaki felt he had often looked at his father negatively and that
he had never said anything "lofty or inspiring". He regretted not having a serious discussion with
his father, and felt he had inherited his "anarchistic feelings and his lack of concern about
embracing contradictions".

-----------------------------------------------------------------

-----------------------------------------------------------------

Some of Miyazaki's earliest memories are of "bombed-out cities". In 1944, when he was three years
old, Miyazaki's family evacuated to Utsunomiya. After the city was bombed in July 1945, they
evacuated to Kanuma. The bombing left a lasting impression on Miyazaki, then aged four. As a child,
he suffered from digestive problems, and was told he would not live beyond 20, making him feel like
an outcast; he considered himself "clumsy and weak", protected at school by his older brother. From
1947 to 1955, Miyazaki's mother Yoshiko suffered from spinal tuberculosis; she spent the first few
years in hospital before being nursed from home, forcing Miyazaki and his siblings to take over
domestic duties. Yoshiko was frugal, and described as a strict, intellectual woman who regularly
questioned "socially accepted norms". She was closest with Miyazaki, and had a strong influence on
him and his later work, inspiring several of his characters. Yoshiko Miyazaki died in July 1983 at
the age of 72. Miyazaki began school as an evacuee in 1947, at an elementary school in Utsunomiya,
completing the first through third grades. After his family moved back to Suginami-ku in 1950,
Miyazaki completed the fourth grade at Ōmiya Elementary School, and fifth grade at Eifuku Elementary
School, which was newly established after splitting off from Ōmiya Elementary. After graduating from
Eifuku as part of the first graduating class, he attended Ōmiya Junior High School. He aspired to
become a manga artist, but discovered he could not draw people; instead, he drew planes, tanks, and
battleships for several years. Miyazaki was influenced by several manga artists, such as Tetsuji
Fukushima, Soji Yamakawa and Osamu Tezuka. Miyazaki destroyed much of his early work, believing it
was "bad form" to copy Tezuka's style as it was hindering his own development as an artist. He
preferred to see artists like Tezuka as fellow artists rather than idols to worship. Around this
time, Miyazaki often saw movies with his father, who was an avid moviegoer; memorable films for
Miyazaki include Meshi (1951) and Tasogare Sakaba (1955). After graduating from Ōmiya Junior High,
Miyazaki attended Toyotama High School. During his third and final year, Miyazaki's interest in
animation was sparked by The White Snake Enchantress (1958), Japan's first feature-length animated
film in color; he had sneaked out to watch the film instead of studying for his entrance exams.
Miyazaki later recounted that, falling in love with its heroine, the film moved him to tears and
left a profound impression, prompting him to create work true to his own feelings instead of
imitating popular trends; he wrote the film's "pure, earnest world" promoted a side of him that
"yearned desperately to affirm the world rather than negate it". After graduating from Toyotama,
Miyazaki attended Gakushuin University in the department of political economy, majoring in Japanese
Industrial Theory; he considered himself a poor student as he instead focused on art. He joined the
"Children's Literature Research Club", the "closest thing back then to a comics club"; he was
sometimes the sole member of the club. In his free time, Miyazaki would visit his art teacher from
middle school and sketch in his studio, where the two would drink and "talk about politics, life,
all sorts of things". Around this time, he also drew manga; he never completed any stories, but
accumulated thousands of pages of the beginnings of stories. He also frequently approached manga
publishers to license their stories. In 1960, Miyazaki was a bystander during the Anpo protests,
having developed an interest after seeing photographs in Asahi Graph; by that point, he was too late
to participate in the demonstrations. Miyazaki graduated from Gakushuin in 1963 with degrees in
political science and economics.

-----------------------------------------------------------------

### Document Embedding

- 준비한 text data(여기선 `paragraph`)를 활용해서 embedding 생성 준비
  - 단순히 `model`에서 `.encode`를 활용하면 됨.

In [11]:
docs_embed = model.encode(paragraphs, normalize_embeddings=True)

C:\Users\skdbs\.cache\huggingface\modules\transformers_modules\Alibaba_hyphen_NLP\new_hyphen_impl\40ced75c3017eb27626c9d4ea981bde21a2662f4\modeling.py:579: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


- 생성된 embedding은 text의 semantic meaning를 포착할 수 있는 dense vector.
  - model이 text를 수학적인 방식으로 형태를 이해하고 처리할 수 있도록 해줌.
- 이제 이 vector를 normalize.
  - 왜 normalize를 할까?
    - 정규화된(길이가 1) vector(embedding)는 vector 간의 거리가 **값 크기 차이보다는 방향 차이를 반영하도록 보장**하기 때문.
    - 즉, **서로 다른 text chunk들이 얼마나 가깝거나 유사한지를 비교하려는 similarity search와 같은 작업에서 모델의 성능을 향상**시킬 수 있음.
- 생성된 `docs_embed`는 text data의 vector representation 모음집이고, 각 vector는 `paragraphs` list에 있는 paragraph에 해당함.

In [12]:
docs_embed.shape

(24, 768)

### Query embedding

- 전체 문서를 embedding으로 표현한 것 처럼, user query를 embedding 해보자.

In [13]:
query = "What was Studio Ghibli's first film?"
query_embed = model.encode(query, normalize_embeddings=True)
query_embed.shape

(768,)

### Query와 가장 유사한 paragraph 탐색

- 가장 유사한 chunk를 가져오는 간단한 방법 중 하나는, 단순히 document embedding과 query embedding을 내적(dot-product)하는 것.
  - 두 vector(or matrix)의 대응되는 요소들을 곱하고 그 결과를 더하는 수학연산으로, 두 vector(or matrix)의 유사도를 측정하는데 주로 사용 됨.

In [14]:
similarites = np.dot(docs_embed, query_embed.T)
print(similarites.shape)
print(similarites)

(24,)

[0.3974758  0.39695853 0.39838523 0.3709961  0.40112233 0.380914
 0.44303593 0.41080886 0.47041556 0.43395585 0.43947777 0.44720432
 0.46445873 0.40873498 0.36114663 0.43905002 0.5042101  0.42784393
 0.40362316 0.41335148 0.34868705 0.39935842 0.2753365  0.39736474]

- `docs_embed`가 `(n_docs, n_dim)`의 형태라면
  - `n_docs`는 document의 수
  - `n_dim`은 각 document embedding의 차원
- 여기서 `query_embed.T`는 vector의 형태를 `(n_dim, 1)`로 변형해서, 차원을 맞춰준 것.
  - `(24, 768)``(768, 1)` 로 변환했으므로, dot product를 하면 `(24,)`가 되는 것.
- dot product 수행 후에 생성된 유사도 행렬은 `(n_docs, )`형태
  - 즉, **`n_docs`개의 요소를 가진 1차원 행렬(vector)**임을 의미.
  - 각 요소는 query와 document 중 하나 사이의 유사도 점수를 나타냄.

<br>

- 그럼 왜 shape를 확인하는 걸까?
  - 최종 형태가 예상대로 생성되었는지 확인하는 것은, dot product가 올바르게 수행되었고 각 문서의 유사도 점수가 계산되었음을 확인 시켜주기 때문.

- 이 두 vector(embedding)간의 dot product 결과 값이 높을수록, 문서 간의 유사성이 높다는 것을 의미.
  - 정규화된 embedding이라면 유사성 값은 vector 간의 cosine similarity에 정비례
  - 정규화되지 않은 경우에도 유사도를 나타내지만, embedding의 크기도 반영함.

- 이제 여기서 가장 유사한 문서 3개를 찾아보자면...

In [15]:
top_3_idx = np.argsort(similarites, axis=0)[-3:][::-1].tolist()
print(top_3_idx)

[16, 8, 12]

- `np.argsort(similarites, axis=0)`은 배열의 index를 오름차순으로 정렬.
  - 예시로, `similarites = [0.1, 0.7, 0.4]`라면, `np.argsort`는 `[0, 2, 1]`을 return함.
- `[-3:]`은 유사도 점수가 가장 높은 상위 3개의 요소(정렬된 후의 마지막 3개 요소)의 index를 선택.
- `[::-1]`은 순서를 reverse해서 index가 유사도의 내림차순으로 정렬.

그래서 가장 유사한 document를 찾아보면...

In [16]:
most_similar_documents = [paragraphs[idx] for idx in top_3_idx]

CONTEXT = ""
for i, p in enumerate(most_similar_documents):
    wrapped_text = textwrap.fill(p, width=100)

    print("-----------------------------------------------------------------")
    print(wrapped_text)
    print("-----------------------------------------------------------------")
    CONTEXT += wrapped_text + "\n\n"

-----------------------------------------------------------------

Themes Miyazaki's works are characterized by the recurrence of themes such as feminism,
environmentalism, pacifism, love, and family. His narratives are also notable for not pitting a hero
against an unsympathetic antagonist; Miyazaki felt Spirited Away's Chihiro "manages not because she
has destroyed the 'evil', but because she has acquired the ability to survive". Miyazaki's films
often emphasize environmentalism and the Earth's fragility. Margaret Talbot stated Miyazaki dislikes
modern technology, and believes much of modern culture is "thin and shallow and fake"; he
anticipates a time with "no more high-rises". Miyazaki felt frustrated growing up in the Shōwa
period from 1955 to 1965 because "nature—the mountains and rivers—was being destroyed in the name of
economic progress". Peter Schellhase of The Imaginative Conservative identified that several
antagonists of Miyazaki's films "attempt to dominate nature in pursuit of political domination, and
are ultimately destructive to both nature and human civilization". Miyazaki is critical of
exploitation under both communism and capitalism, as well as globalization and its effects on modern
life, believing "a company is common property of the people that work there". Ram Prakash Dwivedi
identified values of Mahatma Gandhi in the films of Miyazaki. Several of Miyazaki's films feature
anti-war themes. Daisuke Akimoto of Animation Studies categorized Porco Rosso as "anti-war
propaganda" and felt the protagonist, Porco, transforms into a pig partly due to his extreme
distaste of militarism. Akimoto also argues that The Wind Rises reflects Miyazaki's "antiwar
pacifism", despite Miyazaki stating that the film does not attempt to "denounce" war. Schellhase
also identifies Princess Mononoke as a pacifist film due to the protagonist, Ashitaka; instead of
joining the campaign of revenge against humankind, as his ethnic history would lead him to do,
Ashitaka strives for peace. David Loy and Linda Goodhew argue both Nausicaä of the Valley of the
Wind and Princess Mononoke do not depict traditional evil, but the Buddhist roots of evil: greed,
ill will, and delusion; according to Buddhism, the roots of evil must transform into "generosity,
loving-kindness and wisdom" in order to overcome suffering, and both Nausicaä and Ashitaka
accomplish this. When characters in Miyazaki's films are forced to engage in violence, it is shown
as being a difficult task; in Howl's Moving Castle, Howl is forced to fight an inescapable battle in
defense of those he loves, and it almost destroys him, though he is ultimately saved by Sophie's
love and bravery. Suzuki described Miyazaki as a feminist in reference to his attitude to female
workers. Miyazaki has described his female characters as "brave, self-sufficient girls that don't
think twice about fighting for what they believe in with all their heart", stating they may "need a
friend, or a supporter, but never a saviour" and "any woman is just as capable of being a hero as
any man". Nausicaä of the Valley of the Wind was lauded for its positive portrayal of women,
particularly protagonist Nausicaä. Schellhase noted the female characters in Miyazaki's films are
not objectified or sexualized, and possess complex and individual characteristics absent from
Hollywood productions. Schellhase also identified a "coming of age" element for the heroines in
Miyazaki's films, as they each discover "individual personality and strengths". Gabrielle Bellot of
The Atlantic wrote that, in his films, Miyazaki "shows a keen understanding of the complexities of
what it might mean to be a woman". In particular, Bellot cites Nausicaä of the Valley of the Wind,
praising the film's challenging of gender expectations, and the strong and independent nature of
Nausicaä. Bellot also noted Princess Mononoke's San represents the "conflict between selfhood and
expression". Miyazaki is concerned with the sense of wonder in young people, seeking to maintain
themes of love and family in his f

-----------------------------------------------------------------

-----------------------------------------------------------------

My Neighbor Totoro and Kiki's Delivery Service (1987–1989) Miyazaki's next film, My Neighbor Totoro,
originated in ideas he had as a child; he felt "Totoro is where my consciousness began". An attempt
to pitch My Neighbor Totoro to Tokuma Shoten in the early 1980s had been unsuccessful, and Miyazaki
faced difficulty in attempting to pitch it again in 1987. Suzuki proposed that Totoro be released as
a double bill alongside Takahata's Grave of the Fireflies; as the latter, based on the 1967 short
story by Akiyuki Nosaka, had historical value, Suzuki predicted school students would be taken to
watch both. Totoro features the theme of the relationship between the environment and humanity,
showing that harmony is the result of respecting the environment. The film also references
Miyazaki's mother; the child protagonists' mother is bedridden. As with Laputa, Miyazaki wrote
lyrics for Totoro's end theme. Miyazaki struggled with the film's script until he read a Mainichi
Graph story about Japan forty years prior, opting to set the film in the country before Tokyo's
expansion and the advent of television. Miyazaki has subsequently donated money and artwork to fund
preservation of the forested land in Saitama Prefecture, in which the film is set. Production of My
Neighbor Totoro began in April 1987 and took exactly a year; it was released on April 16, 1988.
While the film received critical acclaim, it was only moderately successful at the box office.
Studio Ghibli approved merchandising rights in 1990, which led to major commercial success;
merchandise profits alone were able to sustain the studio for years. The film was labeled a cult
classic, eventually gaining success in the United States after its release in 1993, where its home
video release sold almost 500,000 copies. Akira Kurosawa said the film moved him, naming it among
his hundred favorite films—one of few Japanese films to be named. An asteroid discovered by Takao
Kobayashi in December 1994 was named after the film: 10160 Totoro. In 1987, Studio Ghibli acquired
the rights to create a film adaptation of Eiko Kadono's novel Kiki's Delivery Service. Miyazaki's
work on My Neighbor Totoro prevented him from directing the adaptation; he acted as producer, while
Sunao Katabuchi was chosen as director and Nobuyuki Isshiki as script writer. Miyazaki's
dissatisfaction of Isshiki's first draft led him to make changes to the project, ultimately taking
the role of director. Kadono expressed her dissatisfaction with the differences between the book and
screenplay, but Miyazaki and Takahata convinced her to let production continue. The film was
originally intended to be a 60-minute special, but expanded into a feature film after Miyazaki
completed the storyboards and screenplay. Miyazaki felt the struggles of the protagonist, Kiki,
reflected the feelings of young girls in Japan yearning to live independently in cities, while her
talents reflected those of real girls, despite her magical powers. In preparation for production,
Miyazaki and other senior staff members traveled to Sweden, where they captured eighty rolls of film
in Stockholm and Visby, the former being the primary inspiration behind the film's city. Kiki's
Delivery Service premiered on July 29, 1989; it was critically successful, winning the Anime Grand
Prix. With more than 2.6 million tickets sold, it earned ¥2.15 billion at the box office and was the
highest-grossing film in Japan in 1989. Miyazaki and Studio Ghibli personally approved the
subsequent English translations.

-----------------------------------------------------------------

-----------------------------------------------------------------

Howl's Moving Castle and Ponyo (2001–2008) Studio Ghibli announced the production of Howl's Moving
Castle in September 2001, based on the novel by Diana Wynne Jones, which Miyazaki had read in 1999.
Toei Animation's Mamoru Hosoda was originally selected to direct the film, but disagreements between
Hosoda and Studio Ghibli executives led to the project's abandonment in 2002. After six months,
Studio Ghibli resurrected the project. Miyazaki was inspired to direct the film, struck by the image
of a castle moving around the countryside, and he took creative liberties in its depiction. Some
computer animation was used to animate the castle's movements, though Miyazaki dictated it consist
of no more than 10 percent of the film. Miyazaki traveled to Colmar and Riquewihr in Alsace, France,
to study the architecture and the surroundings for the film's setting, while additional inspiration
came from the concepts of future technology in Albert Robida's work. The war featured in the film
was thematically influenced by the 2003 invasion and subsequent war in Iraq, the events of which
enraged Miyazaki. Howl's Moving Castle was released on November 20, 2004, and received widespread
critical acclaim. The film received the Golden Osella for Technical Excellence at the 61st Venice
International Film Festival, and was nominated for the Academy Award for Best Animated Feature. In
Japan, the film sold more than 1.1 million tickets within two days and grossed a record US$14.5
million in its first week. It became Japan's third-highest-grossing film, and remains among the top
rankings with a worldwide gross of over ¥19.3 billion. Miyazaki received the honorary Golden Lion
for Lifetime Achievement award at the 62nd Venice International Film Festival in 2005. He visited
the United States in June 2005 to promote the film. In March 2005, Studio Ghibli split from Tokuma
Shoten, and Miyazaki became corporate director. After Howl's Moving Castle, Miyazaki created some
short films for the Ghibli Museum, for which he returned solely to traditional animation techniques;
all three began screening in January 2006. Studio Ghibli obtained the rights to produce an
adaptation of Ursula K. Le Guin's Earthsea novels in 2003; Miyazaki had contacted her in the 1980s
expressing interest but she declined, unaware of his work. Upon watching My Neighbor Totoro several
years later, she expressed approval to the concept and met with Suzuki in August 2005, who wanted
Miyazaki's son Goro to direct the film, as Miyazaki had wished to retire. Disappointed that Miyazaki
was not directing but under the impression he would supervise his son's work, Le Guin approved of
the film's production. Miyazaki later publicly opposed and criticized Goro's appointment as
director. The film's designs were partly inspired by Miyazaki's manga The Journey of Shuna. During a
screening of the film, Miyazaki commented, "You shouldn't make a picture based on your emotions". He
later wrote a message for his son: "It was made honestly, so it was good". In February 2006,
Miyazaki traveled to the United Kingdom to research A Trip to Tynemouth (based on Robert Westall's
"Blackham's Wimpy"), for which he designed the cover, created a short manga, and worked as editor;
it was released in October. Miyazaki's next film, Ponyo, began production in May 2006. It was
initially inspired by "The Little Mermaid" by Hans Christian Andersen, though began to take its own
form as production continued. Miyazaki aimed for the film to celebrate the innocence and
cheerfulness of a child's universe. He was intimately involved with the artwork, preferring to draw
the sea and waves himself, as he enjoyed experimenting. Two short films—Looking for a Home and Water
Spider Monmon—were made for the Ghibli Museum shortly before Ponyo entered production as animation
experiments for sea life. Ponyo features 170,000 frames—a record for Miyazaki. Its seaside village
was inspired by Tomonoura, a town in Setonaikai National Park, where Miy

-----------------------------------------------------------------

### Reponse 생성

- 이제 query도 있고, 유사한 chunk도 찾았으니 LLM에 전달해주면 된다.

In [17]:
query = "What was Studio Ghibli's first film?"

# prompt 준비
prompt = f"""
use the following CONTEXT to answer the QUESTION at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

CONTEXT: {CONTEXT}
QUESTION: {query}

"""

~~OpenAI API를 활용해보자~~


(일정 credit 과금해야 하니까 Gemini 쓰자)
- 여기서 API key는 환경변수로 저장함.

[GEMINI API docs](https://ai.google.dev/gemini-api/docs/quickstart?hl=ko)

In [18]:

import google.genai as genai


# Gemini API 설정
client = genai.Client(api_key = os.environ.get("GEMINI_API_KEY"))

# model 설정 & response 설정
response = client.models.generate_content(
    model = 'gemini-2.5-flash',
    contents = prompt
)

print(response.text)

I don't know the answer. The provided text does not explicitly state which film was Studio Ghibli's first.

- 여기서 model은 `gemini-2.5-flash`를 사용.
- `contents`인자가 이제 LLM에게 건네줄 prompt.
  - CONTEXT와 query를 주어줬으니, 이를 활용해서 답변을 생성함.